# 实验五 · 奇偶换位排序 —— 并行区域的复用与同步粒度

**所属**：《并行计算技术》第五章 · OpenMP 编程　|　**难度**：⭐⭐⭐ 重点　|　**预计时长**：30–40 分钟

实验四考察的是并行区域**内部**：线程组建立之后，迭代如何在各线程之间分配。那里的性能损失有两个来源——负载不均衡造成的等待，以及调度机制自身的代价。

本实验把考察对象换成并行区域的**入口与出口**，以及相位之间的**栅栏**。无论区域内部做什么，每一次进入并退出并行区域，都要付出线程组创建与销毁的代价；而每一个工作分担构造的末尾，都有一次栅栏同步。奇偶换位排序需执行 $n$ 个相位：若每个相位均创建一次线程组，则前一项代价将累积为单次的约 $n$ 倍；后一项则无论如何都要发生 $n$ 次。本实验通过三个版本分别度量这两项代价，并给出两种不同的应对方法。

> **实验说明**
> 1. 本实验采用**递进式的版本组织**：以串行实现为基准，每个版本仅引入一种新的 OpenMP 构造或一处相应的代码改写，并在同一次运行中完成全部版本的计时与正确性校验，因而各版本面对的是完全相同的数据与运行环境，各版本之间具有可比性。
> 2. 请自上而下依次执行各单元格（Shift+Enter）。
> 3. 本实验依赖支持 OpenMP 的 **GCC 编译器**，建议在华为鲲鹏处理器或其他 AArch64 平台上运行。
> 4. 遇到 🔧 **动手练习** 与 🤔 **思考题** 时，建议先独立完成，再阅读后续内容。
> 5. V3 改变的是**算法本身**（块内改用 `qsort`），其加速比同时包含算法收益与并行收益，不能与 V1、V2 直接对比。第 11 节将专门讨论这一点。
> 6. 命令行参数 `n` 必须能被线程数整除。建议取 10080，该数可被 1 至 8 整除，便于做线程数扩展性实验。

## 🎯 学习目标

完成本实验后，学生应能够：

- 复述奇偶换位排序的算法流程，说明其为何具有良好的并行性
- 区分 `#pragma omp parallel` 与 `#pragma omp for` 的职责，掌握把二者**分开书写**的场合与写法
- 定量认识 **Fork-Join 开销**：能够设计实验测出单次并行区域创建的代价
- 理解 `#pragma omp for` 末尾的**隐式栅栏**如何保证相位之间的次序，以及 `nowait` 子句在何种条件下可以安全去除它
- 掌握 `single` 与 `master` 构造的区别，说明何时必须使用 `single`
- 理解**并行粒度**这一概念：以粗粒度的独立工作替代细粒度的频繁同步，往往比优化同步机制本身收益更大
- 区分「并行加速」与「算法改进」两类收益，避免在性能报告中混淆二者

## 🗺️ 学习路径

1. **准备阶段**：理解奇偶换位排序的相位结构与比较交换模式
2. **V0 · 串行基准**：$O(n^2)$ 的比较交换
3. **V1 · 每相位一个并行区域**：最直接的并行化，暴露 Fork-Join 开销
4. **V2 · 全程一个并行区域**：把 `parallel` 提到相位循环之外，由 `#pragma omp for` 的隐式栅栏维持相位次序
5. **V3 · 块分解 + 归并交换**：改变并行粒度，块内 `qsort`，块间归并划分
6. **可视化与分析**：分离 Fork-Join 开销、同步开销与算法收益

## 1. 背景知识：奇偶换位排序

### 1.1 算法描述

奇偶换位排序（odd-even transposition sort）是冒泡排序的一种变体。其绝对效率并不突出，但结构上具有**高度的可并行性**。

算法执行 $n$ 个相位，相位按奇偶交替：

- **偶相位**：比较并交换下标对 $(0,1), (2,3), (4,5), \ldots$
- **奇相位**：比较并交换下标对 $(1,2), (3,4), (5,6), \ldots$

```text
  初始:   5   2   8   1   9   3

  偶相位: [5   2] [8   1] [9   3]     每对独立，可完全并行
          → 2   5   1   8   3   9

  奇相位:  2  [5   1] [8   3]  9      每对独立，可完全并行
          → 2   1   5   3   8   9

  偶相位: [2   1] [5   3] [8   9]
          → 1   2   3   5   8   9
```

### 1.2 为何适合并行

**同一相位内的所有比较交换对彼此不相交**：偶相位涉及的下标对是$(0,1), (2,3), \ldots$，任意两对之间没有公共下标，因而可以由不同线程同时执行，不存在数据竞争。

但**相位之间必须严格串行**：第 $k+1$ 相位读取的正是第 $k$ 相位写入的数据。这是一个典型的**批量同步**（bulk-synchronous）结构：

```text
  相位0: ████████  ← 并行
         ─────────  ← 栅栏
  相位1: ████████  ← 并行
         ─────────  ← 栅栏
  相位2: ████████
         ...
```

**关键结论**：这类结构需要 $n$ 次栅栏同步。如何以较低的代价提供这 $n$ 次同步，是本实验的核心问题。

### 1.3 算法复杂度

$n$ 个相位、每相位约 $n/2$ 次比较，总计 $O(n^2)$。这一复杂度远高于 `qsort` 的 $O(n \log n)$，因此本实验的 V0 至 V2 在绝对性能上并无实用价值。

它们的价值在于：三个版本执行的是**完全相同的算法**，因而三者之间的耗时差异可归因于并行结构本身。V3 则提供了另一重对照：当并行结构本身已无明显改进空间时，性能瓶颈往往转移到算法层面。

## 2. 并行区域的结构选择

### 2.1 `parallel` 与 `for` 的分离书写

前面几个实验均采用合写形式 `#pragma omp parallel for`。实际上，二者是**两个独立的构造**，可以分开书写：

```c
#pragma omp parallel num_threads(p)      // 构造一：创建线程组
{
  for (phase = 0; phase < n; phase++) {  // 普通循环，每个线程都完整执行
#pragma omp for                          // 构造二：分担紧随其后的循环
    for (i = 1; i < n; i += 2) { ... }
  }
}
```

分开书写的意义在于：**线程组只需创建一次，即可被多个工作分担构造重复使用**。外层的相位循环由每个线程各自完整执行（这正是实验一所指出的：`parallel` 构造使线程组中的各个线程执行同一个结构化块，而不划分工作），而内层的比较交换循环则由 `#pragma omp for` 划分给各线程分担。

| 写法 | 线程组创建次数 | 栅栏次数 |
|---|---|---|
| 每相位一个 `parallel for` | $n$ | $n$ |
| 一个 `parallel` + $n$ 个 `for` | **1** | $n$ |

栅栏次数无法减少，这是算法本身的要求；但线程组的创建次数可以从 $n$ 降到 1。

### 2.2 Fork-Join 开销的量级

创建一个线程组需要：分配线程栈、初始化线程私有数据结构、唤醒工作线程、分配任务、以及区域结束时的汇合。在 Linux + libgomp 上，这一代价通常在**微秒量级**。

微秒量级的开销在单次进入时并不显著，但在高频进入的情形下需结合具体问题规模加以衡量：

$$\text{总开销} = n \times t_{\text{fork-join}}$$

当 $n = 10080$、$t_{\text{fork-join}} \approx 5\,\mu s$ 时，总开销约为 50 毫秒——而整个串行排序也不过数十毫秒。**此时开销与有效计算处于同一量级，甚至可能超过有效计算本身**。

> 值得注意的是：libgomp 会**复用**空闲的工作线程，因此第二次及以后进入并行区域的代价低于第一次。但即便如此，每次进入仍需分配任务与同步，代价并非为零。

## 3. OpenMP 关键知识点

### 3.1 隐式栅栏与 `nowait`

以下四个构造的末尾都存在隐式栅栏：

| 构造 | 末尾是否有隐式栅栏 | 能否用 `nowait` 去除 |
|---|---|---|
| `#pragma omp parallel` | 有 | **不能** |
| `#pragma omp for` | 有 | 能 |
| `#pragma omp sections` | 有 | 能 |
| `#pragma omp single` | 有 | 能 |

`nowait` 的写法：

```c
#pragma omp for nowait
for (i = 0; i < n; i++) { ... }
// 先完成的线程不等待，直接向下执行
```

**本实验不能使用 `nowait`**。相位之间存在真实的数据依赖：若线程 A 已经进入相位 $k+1$ 并开始读取数据，而线程 B 仍在相位 $k$ 写入同一批数据，两者将构成数据竞争，结果不再正确。

> **`nowait` 的适用条件**：紧随其后的代码不读取本循环写入的数据。典型场景是两个彼此无关的循环相邻出现，此时去掉中间的栅栏可以让先完成的线程提前开始第二个循环。

### 3.2 `single` 与 `master`

并行区域内若有某段代码只需执行一次（例如打印进度、更新全局计数），可用以下两个构造之一：

| 构造 | 由谁执行 | 末尾是否有隐式栅栏 | 能否用 `nowait` |
|---|---|---|---|
| `#pragma omp single` | **任意一个**先到达的线程 | 有 | 能 |
| `#pragma omp master` | **必定**是 0 号线程 | **无** | 不适用 |

> **关于 `master` 的规范时效性**：`master` 构造在 OpenMP 5.1 中已被列为**弃用特性**，由 `masked` 构造取代。`masked` 的默认语义与 `master` 相同，并可通过 `filter(n)` 子句指定由编号为 `n` 的线程执行。本实验沿用 `master` 以配合当前编译器的普遍支持情况；新编写的代码建议使用 `masked`。

两者的区别不仅在于「谁来执行」，更在于**是否同步**：

```c
#pragma omp parallel
{
#pragma omp single
  { init(); }          // 有隐式栅栏：其余线程等待 init 完成
  compute();           // 因而可以安全地使用 init 的结果
}

#pragma omp parallel
{
#pragma omp master
  { init(); }          // 无栅栏：其余线程不等待
  compute();           // ⚠️ 可能在 init 完成之前就开始执行
}
```

**选择准则**：若后续代码依赖该段代码的结果，应使用 `single`；若仅为与计算无关的操作（如打印日志），且不希望引入同步，则可使用 `master`。

实验八生成任务树时会用到 `single`，届时会再次讨论。

### 3.3 `default(none)` 与显式数据环境

本实验的 V1 与 V2 全部采用 `default(none)` 并逐一列出变量属性：

```c
#pragma omp parallel num_threads(thread_count) default(none) \
    shared(a, n) private(phase, i, tmp)
```

注意 `phase`、`i`、`tmp` 三个变量都声明在函数体内、并行区域**之外**，按实验二的默认规则它们本应是**共享**的。若不显式私有化，三个线程会共用同一个循环变量，结果必然错误。这正是 `default(none)` 的价值：它强制程序员为每一个变量显式声明其数据共享属性。

## 4. 环境准备与检查

本节确认三项内容：编译器是否支持 OpenMP、运行时报告的处理器数量、以及各处理器核心的最大频率是否一致。

第三项检查针对**异构多核**平台。Arm 的 big.LITTLE 架构把高性能核心与高能效核心集成在同一块芯片上，二者的频率与微架构均不相同。在这类平台上，同一段代码在不同类型的核心上执行，耗时可能相差 2 倍以上，线程数与加速比之间因而不再是简单的线性关系。

需要说明的是，最大频率不一致只是异构多核的**必要非充分**证据：同构多核平台也可能因加速频率（boost）策略或芯片分级（binning）而上报不同的 `cpuinfo_max_freq`。因此下面的检查只给出提示，确认平台是否为异构架构还需结合 `lscpu` 输出的核心型号信息。

In [ ]:
import os
import re
import subprocess
import platform

print('=' * 60)
print(' 一、平台信息')
print('=' * 60)
print('操作系统   :', platform.system(), platform.release())
print('处理器架构 :', platform.machine())
print('逻辑核心数 :', os.cpu_count())

print()
print('=' * 60)
print(' 二、编译器与 OpenMP 支持')
print('=' * 60)
gcc_ver = subprocess.run(['gcc', '--version'], capture_output=True,
                         text=True).stdout.splitlines()[0]
print('编译器     :', gcc_ver)

probe = subprocess.run('echo | gcc -fopenmp -dM -E -x c - | grep _OPENMP',
                       shell=True, capture_output=True, text=True).stdout.strip()
if probe:
    ver = int(probe.split()[-1])
    spec = {200805: '3.0', 201107: '3.1', 201307: '4.0',
            201511: '4.5', 201811: '5.0', 202011: '5.1'}.get(ver, '未知')
    print('_OPENMP    :', ver, '(对应 OpenMP %s 规范)' % spec)
    print('数组段归约 :', '支持' if ver >= 201511 else '不支持（需要 4.5 及以上）')
else:
    print('⚠️  未检测到 OpenMP 支持，请确认编译时带有 -fopenmp')

print()
print('=' * 60)
print(' 三、核心频率与异构性检查')
print('=' * 60)
freqs = []
for cpu in range(os.cpu_count() or 1):
    path = '/sys/devices/system/cpu/cpu%d/cpufreq/cpuinfo_max_freq' % cpu
    try:
        with open(path) as f:
            freqs.append((cpu, int(f.read().strip()) // 1000))
    except OSError:
        pass

if not freqs:
    print('无法读取 cpufreq 节点，跳过异构性检查。')
else:
    for cpu, mhz in freqs:
        print('  CPU%-2d 最大频率: %5d MHz' % (cpu, mhz))
    distinct = sorted(set(m for _, m in freqs))
    if len(distinct) > 1:
        print()
        print('⚠️  检测到 %d 种不同的最大频率，本平台可能为异构多核架构（如 Arm big.LITTLE）。'
              % len(distinct))
        print('    请结合 lscpu 输出的核心型号信息进一步确认。')
        print('    若确为异构平台，测速前建议执行：')
        print('      export OMP_PROC_BIND=close')
        print('      export OMP_PLACES=cores')
    else:
        print()
        print('✅ 全部核心的最大频率一致，可按同构多核平台处理。')

print()
print('OMP_NUM_THREADS =', os.environ.get('OMP_NUM_THREADS', '（未设置，由运行时决定）'))
print('OMP_PROC_BIND   =', os.environ.get('OMP_PROC_BIND', '（未设置）'))
print('OMP_PLACES      =', os.environ.get('OMP_PLACES', '（未设置）'))

## 5. 实验工具函数

本节定义三个贯穿全章的辅助函数，后续各实验均直接调用，不再重复说明。

| 函数 | 作用 |
|---|---|
| `compile_c(src)` | 以 `-O3 -fopenmp -Wall -Wextra` 编译指定源文件，并回显全部告警 |
| `run_c(binary, *args)` | 运行可执行文件并原样打印其标准输出 |
| `parse_table(output)` | 从程序输出的结果表中提取「方法名 / 耗时 / 加速比 / 校验」四列 |

**关于编译选项**：全章统一使用 `-O3 -fopenmp`。AArch64 平台的 NEON 属于基线指令集，无需附加 `-march` 或 `-mcpu` 选项。`-Wall -Wextra` 用于暴露数据环境声明不当引发的告警，这类告警在 OpenMP 程序中往往是并发缺陷的征兆，不应忽略。

In [ ]:
import subprocess
import re
import os

SRC_DIR = 'src_oddeven_sort'
os.makedirs(SRC_DIR, exist_ok=True)

CFLAGS = ['-O3', '-fopenmp', '-Wall', '-Wextra']


def compile_c(src, extra=('-lm',)):
    """编译单个源文件，返回可执行文件路径；编译失败时抛出异常。"""
    src_path = os.path.join(SRC_DIR, src)
    binary = os.path.join(SRC_DIR, os.path.splitext(src)[0])
    cmd = ['gcc'] + CFLAGS + ['-o', binary, src_path] + list(extra)
    print('$', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.stdout.strip():
        print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print(proc.stderr.rstrip())
    if proc.returncode != 0:
        raise RuntimeError('编译失败：%s' % src)
    print('✅ 编译通过，无告警' if not proc.stderr.strip()
          else '⚠️  编译通过，但存在告警，请逐条阅读')
    return binary


def run_c(binary, *args, env=None):
    """运行可执行文件，打印并返回其标准输出。"""
    cmd = [binary] + [str(a) for a in args]
    print('$', ' '.join(cmd))
    print()
    run_env = dict(os.environ)
    if env:
        run_env.update({k: str(v) for k, v in env.items()})
    proc = subprocess.run(cmd, capture_output=True, text=True, env=run_env)
    print(proc.stdout.rstrip())
    if proc.stderr.strip():
        print('[stderr]', proc.stderr.rstrip())
    return proc.stdout


ROW_RE = re.compile(r'^\|\s*(.+?)\s*\|\s*([0-9.]+)\s*\|\s*([0-9.]+)x\s*\|\s*(\S+)\s*\|$')


def parse_table(output):
    """解析结果表，返回 [(方法名, 耗时ms, 加速比, 校验结论), ...]。"""
    rows = []
    for line in output.splitlines():
        m = ROW_RE.match(line.strip())
        if m:
            rows.append((m.group(1), float(m.group(2)),
                         float(m.group(3)), m.group(4)))
    return rows


print('工具函数已就绪，源码目录：', os.path.abspath(SRC_DIR))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

matplotlib.rcParams['font.sans-serif'] = ['DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False

C_BASE, C_GOOD, C_FAIL, C_SLOW = '#7f7f7f', '#1f77b4', '#d62728', '#ff7f0e'


def plot_speedup(rows, title, figsize=(10, 5)):
    """绘制加速比柱状图。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。"""
    if not rows:
        print('未解析到结果行，请先运行上一单元格。')
        return
    names = [r[0] for r in rows]
    speeds = [r[2] for r in rows]
    colors = []
    for i, (_, _, sp, chk) in enumerate(rows):
        if i == 0:
            colors.append(C_BASE)
        elif chk == 'FAIL':
            colors.append(C_FAIL)
        elif sp < 1.0:
            colors.append(C_SLOW)
        else:
            colors.append(C_GOOD)

    fig, ax = plt.subplots(figsize=figsize)
    bars = ax.bar(range(len(names)), speeds, color=colors,
                  edgecolor='black', linewidth=0.6, width=0.6)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1.0, alpha=0.7)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=20, ha='right', fontsize=9)
    ax.set_ylabel('Speedup vs. Serial Baseline')
    ax.set_title(title, fontsize=12, pad=12)
    ax.grid(axis='y', linestyle=':', alpha=0.5)
    ax.set_axisbelow(True)

    for bar, (_, ms, sp, chk) in zip(bars, rows):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.02,
                '%.2fx\n%.1f ms%s' % (sp, ms, '' if chk in ('-', 'PASS') else '\nFAIL'),
                ha='center', va='bottom', fontsize=8)

    ax.set_ylim(0, max(speeds) * 1.30)
    plt.tight_layout()
    plt.show()


print('绘图函数已就绪。配色：灰=基准，蓝=有效加速，橙=慢于基准，红=校验失败。')

## 6. 版本设计总览

| 版本 | 新增计算函数 | 算法 | 线程组创建次数 | 结果表行号 |
|---|---|---|---|---|
| **V0** | `oddeven_serial` | $O(n^2)$ 奇偶换位 | 0 | 1 |
| **V1** | `oddeven_multi_region` | 同上 | **$n$** | 2 |
| **V2** | `oddeven_single_region` | 同上 | **1** | 3 |
| **V3** | `oddeven_blocked` | $O(n\log n)$ 块内排序 + 归并划分 | 2 | 4 |

**三段对照的读法**：

| 对照 | 差异 | 度量的对象 |
|---|---|---|
| V0 → V1 | 引入并行 | 并行收益 **减去** Fork-Join 开销 |
| V1 → V2 | 线程组创建次数 $n \to 1$ | **Fork-Join 开销本身** |
| V2 → V3 | 算法与并行粒度 | 算法收益 **加上** 粒度收益 |

第二段对照是本实验的核心。由于 V1 与 V2 的计算量、同步次数完全相同，它们的耗时之差**主要**来自线程组的反复创建与销毁。

## 7. 逐版本代码讲解

### 7.1 V0 · 串行基准

```c
for (long phase = 0; phase < n; phase++) {
  if (phase % 2 == 0) {
    for (long i = 1; i < n; i += 2) {
      if (a[i - 1] > a[i]) { /* 交换 */ }
    }
  } else {
    for (long i = 1; i < n - 1; i += 2) {
      if (a[i] > a[i + 1]) { /* 交换 */ }
    }
  }
}
```

注意两个内层循环的边界不同：偶相位比较 $(i-1, i)$，$i$ 可取到 $n-1$；奇相位比较 $(i, i+1)$，$i$ 至多取到 $n-2$。

### 7.2 V1 · 每相位一个并行区域

```c
for (long phase = 0; phase < n; phase++) {
  if (phase % 2 == 0) {
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(a, n) private(i, tmp)
    for (i = 1; i < n; i += 2) { ... }
  } else {
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(a, n) private(i, tmp)
    for (i = 1; i < n - 1; i += 2) { ... }
  }
}
```

这是最直接的并行化：在每个内层循环前加一行制导语句。代码改动最小，且不影响正确性。

**代价**：相位循环执行 $n$ 次，每次都完整地创建并销毁一个线程组。以 $n = 10080$ 计，这意味着一万余次 Fork-Join。

### 7.3 V2 · 全程一个并行区域

```c
#pragma omp parallel num_threads(thread_count) default(none) \
    shared(a, n) private(phase, i, tmp)
for (phase = 0; phase < n; phase++) {       // 每个线程都执行这个循环
  if (phase % 2 == 0) {
#pragma omp for
    for (i = 1; i < n; i += 2) { ... }
  } else {
#pragma omp for
    for (i = 1; i < n - 1; i += 2) { ... }
  }
  // #pragma omp for 末尾的隐式栅栏保证：所有线程完成相位 k
  // 之后，才会有线程进入相位 k+1
}
```

**以下三点需要理解**：

1. **`#pragma omp parallel` 后面直接跟一条 `for` 语句**，而非花括号包裹的复合语句。这是合法的——`for` 语句本身就是一个结构化块。
2. **相位循环被每个线程完整执行**。各线程都完整执行一遍 `phase` 由 0 至 $n-1$ 的循环，但每一轮中真正的比较交换工作由 `#pragma omp for` 划分给各线程分担。
3. **`phase` 必须是私有的**。若它是共享的，各线程将共用同一个循环变量，相位次序随即被破坏。

   需要强调的是，其后果不止于计数器竞争。OpenMP 规范要求线程组内的所有线程以**相同的次序**遇到**相同的**工作分担构造；若各线程读到的 `phase` 不同，它们可能走进不同的分支、遇到不同的 `#pragma omp for`，甚至遇到的次数也不相同。这已经违反规范，属于**未定义行为**，实际运行中常表现为部分线程停在隐式栅栏处无限等待，即**死锁**。这一点是思考题 1 的关键。

**正确性的保证**：`#pragma omp for` 末尾的隐式栅栏。所有线程必须在此汇合，然后才能一起进入下一相位。**这正是本实验不能使用 `nowait` 的原因。**

### 7.4 V3 · 块分解 + 归并交换

前三个版本都在优化「如何更快地执行同一个算法」。V3 则从另一个层面入手：**改变并行的粒度**。

```c
// 第一步：每个线程独立排序自己的块，全程无通信
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(a, local_n, thread_count)
for (int p = 0; p < thread_count; p++) {
  qsort(a + (long)p * local_n, (size_t)local_n, sizeof(int), compare_ints);
}

// 第二步：在「块」这一级别上做奇偶换位，共 thread_count 个相位
#pragma omp parallel num_threads(thread_count) default(none) \
    shared(a, local_n, thread_count, temp)
{
  int my_rank = omp_get_thread_num();
  int *my_buf = temp + (long)my_rank * 2 * local_n;   // 每线程独占缓冲区

  for (int phase = 0; phase < thread_count; phase++) {
    if (phase % 2 == 0) {
#pragma omp for
      for (int p = 0; p < thread_count - 1; p += 2) {
        merge_split(a + p * local_n, a + (p + 1) * local_n, local_n, my_buf);
      }
    } else { /* 奇相位，p 从 1 起 */ }
  }
}
```

**`merge_split` 的作用**：把两个各自有序的相邻块归并，再把归并结果的前一半放回左块、后一半放回右块。这样两个块各自仍然有序，且左块的所有元素都不大于右块的任一元素。

```
  左块 [2 5 8]   右块 [1 3 9]
        ↓ 归并
      [1 2 3 5 8 9]
        ↓ 前一半回左块，后一半回右块
  左块 [1 2 3]   右块 [5 8 9]
```

**三项收益**：

| 项 | V1/V2 | V3 |
|---|---|---|
| 相位数 | $n = 10080$ | $p = 4$ |
| 栅栏次数 | $n$ | $p$ |
| 块内算法 | $O(n^2)$ 冒泡 | $O(\frac{n}{p}\log\frac{n}{p})$ 快排 |

**关于每线程独占缓冲区**：`my_buf` 由线程编号索引，各线程的写入区间互不重叠，因而无需任何同步。

> **关于 `n` 必须被线程数整除**：V3 假设各块长度相等，若不整除，末尾元素将被遗漏。因此程序在参数检查阶段即拒绝不能整除的输入并给出提示。这是一种常见的处理方式：与其在算法内部引入复杂的边界处理，不如在入口处明确约束输入；其代价是牺牲了一部分通用性。练习 6 可在此基础上讨论如何放宽这一限制。

## 8. 源代码写入

In [ ]:
%%writefile {SRC_DIR}/omp_oddeven_sort.c
#define _POSIX_C_SOURCE 200809L

#include <omp.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

#ifndef _OPENMP
#error "OpenMP is required. Please compile with -fopenmp."
#endif

#define NTIMES 3
#define MAX_THREADS 16
#define ALIGN_BYTES 64

#define BANNER "============================================================"
#define LINE "------------------------------------------------------------"

// ----------------------------------------------------------------------------
// Common helpers
// ----------------------------------------------------------------------------
static double get_time_ms(void) {
  struct timespec ts;
  clock_gettime(CLOCK_MONOTONIC, &ts);
  return (double)ts.tv_sec * 1000.0 + (double)ts.tv_nsec / 1000000.0;
}

static void *alloc_aligned(size_t bytes) {
  size_t rounded = ((bytes + ALIGN_BYTES - 1) / ALIGN_BYTES) * ALIGN_BYTES;
  return aligned_alloc(ALIGN_BYTES, rounded);
}

static int check_equal_int(const int *ref, const int *test, long n) {
  for (long i = 0; i < n; i++) {
    if (ref[i] != test[i]) {
      return 0;
    }
  }
  return 1;
}

static void print_table_header(void) {
  printf("\n%s\n", LINE);
  printf("| %-26s | %9s | %7s | %-5s |\n", "Method", "Time(ms)", "Speedup",
         "Check");
  printf("|----------------------------|-----------|---------|-------|\n");
}

static void print_row(const char *name, double time_ms, double base_ms,
                      int check) {
  const char *status = (check < 0) ? "-" : (check ? "PASS" : "FAIL");
  double speedup = (time_ms > 0.0) ? base_ms / time_ms : 0.0;
  printf("| %-26s | %9.3f | %6.2fx | %-5s |\n", name, time_ms, speedup, status);
}

static int compare_ints(const void *p, const void *q) {
  int x = *(const int *)p;
  int y = *(const int *)q;
  return (x > y) - (x < y);
}

// ============================================================================
// V0: Serial baseline
// ============================================================================
static void oddeven_serial(int *a, long n) {
  int tmp;

  for (long phase = 0; phase < n; phase++) {
    if (phase % 2 == 0) {
      for (long i = 1; i < n; i += 2) {
        if (a[i - 1] > a[i]) {
          tmp = a[i];
          a[i] = a[i - 1];
          a[i - 1] = tmp;
        }
      }
    } else {
      for (long i = 1; i < n - 1; i += 2) {
        if (a[i] > a[i + 1]) {
          tmp = a[i];
          a[i] = a[i + 1];
          a[i + 1] = tmp;
        }
      }
    }
  }
}

// ============================================================================
// V1: a new team of threads is created and destroyed on every phase.
// ============================================================================
static void oddeven_multi_region(int *a, long n, int thread_count) {
  long i;
  int tmp;

  for (long phase = 0; phase < n; phase++) {
    if (phase % 2 == 0) {
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(a, n) private(i, tmp)
      for (i = 1; i < n; i += 2) {
        if (a[i - 1] > a[i]) {
          tmp = a[i];
          a[i] = a[i - 1];
          a[i - 1] = tmp;
        }
      }
    } else {
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(a, n) private(i, tmp)
      for (i = 1; i < n - 1; i += 2) {
        if (a[i] > a[i + 1]) {
          tmp = a[i];
          a[i] = a[i + 1];
          a[i + 1] = tmp;
        }
      }
    }
  }
}

// ============================================================================
// V2: the team is created once. #pragma omp for distributes the iterations of
// each phase, and its implicit barrier keeps the phases in order.
// ============================================================================
static void oddeven_single_region(int *a, long n, int thread_count) {
  long phase, i;
  int tmp;

#pragma omp parallel num_threads(thread_count) default(none) \
    shared(a, n) private(phase, i, tmp)
  for (phase = 0; phase < n; phase++) {
    if (phase % 2 == 0) {
#pragma omp for
      for (i = 1; i < n; i += 2) {
        if (a[i - 1] > a[i]) {
          tmp = a[i];
          a[i] = a[i - 1];
          a[i - 1] = tmp;
        }
      }
    } else {
#pragma omp for
      for (i = 1; i < n - 1; i += 2) {
        if (a[i] > a[i + 1]) {
          tmp = a[i];
          a[i] = a[i + 1];
          a[i + 1] = tmp;
        }
      }
    }
    // The implicit barrier at the end of #pragma omp for guarantees that every
    // thread finishes phase k before any thread enters phase k + 1.
  }
}

// ============================================================================
// V3: merge two sorted blocks of equal length, then keep the lower half in the
// left block and the upper half in the right block.
// ============================================================================
static void merge_split(int *left, int *right, long len, int *buf) {
  long i = 0;
  long j = 0;
  long k = 0;

  while (i < len && j < len) {
    if (left[i] <= right[j]) {
      buf[k++] = left[i++];
    } else {
      buf[k++] = right[j++];
    }
  }
  while (i < len) {
    buf[k++] = left[i++];
  }
  while (j < len) {
    buf[k++] = right[j++];
  }

  memcpy(left, buf, (size_t)len * sizeof(int));
  memcpy(right, buf + len, (size_t)len * sizeof(int));
}

static void oddeven_blocked(int *a, long n, int thread_count, int *temp) {
  long local_n = n / thread_count;

  // Step 1: every thread sorts its own block, with no communication at all.
#pragma omp parallel for num_threads(thread_count) default(none) \
    shared(a, local_n, thread_count)
  for (int p = 0; p < thread_count; p++) {
    qsort(a + (long)p * local_n, (size_t)local_n, sizeof(int), compare_ints);
  }

  // Step 2: odd-even transposition at block granularity, thread_count phases.
#pragma omp parallel num_threads(thread_count) default(none) \
    shared(a, local_n, thread_count, temp)
  {
    int my_rank = omp_get_thread_num();
    int *my_buf = temp + (long)my_rank * 2 * local_n;

    for (int phase = 0; phase < thread_count; phase++) {
      if (phase % 2 == 0) {
#pragma omp for
        for (int p = 0; p < thread_count - 1; p += 2) {
          merge_split(a + (long)p * local_n, a + (long)(p + 1) * local_n,
                      local_n, my_buf);
        }
      } else {
#pragma omp for
        for (int p = 1; p < thread_count - 1; p += 2) {
          merge_split(a + (long)p * local_n, a + (long)(p + 1) * local_n,
                      local_n, my_buf);
        }
      }
    }
  }
}

int main(int argc, char *argv[]) {
  if (argc != 3) {
    printf("Usage: %s <n> <thread_count>\n", argv[0]);
    printf("Example: %s 10080 4\n", argv[0]);
    return 1;
  }

  long n = strtol(argv[1], NULL, 10);
  int thread_count = (int)strtol(argv[2], NULL, 10);

  if (n <= 0) {
    printf("Error: n must be > 0\n");
    return 1;
  }
  if (thread_count < 1 || thread_count > MAX_THREADS) {
    printf("Error: thread_count must be between 1 and %d\n", MAX_THREADS);
    return 1;
  }
  if (n % thread_count != 0) {
    printf("Error: n must be divisible by thread_count.\n");
    printf("       V3 assigns one contiguous block per thread.\n");
    return 1;
  }

  printf("%s\n", BANNER);
  printf(" Lab 5: Odd-Even Transposition Sort\n");
  printf(" n: %ld | Threads: %d | Block: %ld | Runs: %d\n", n, thread_count,
         n / thread_count, NTIMES);
  printf(" _OPENMP: %d | Procs: %d\n", _OPENMP, omp_get_num_procs());
  printf("%s\n", BANNER);

  size_t bytes = (size_t)n * sizeof(int);
  int *a_orig = (int *)alloc_aligned(bytes);
  int *a_work = (int *)alloc_aligned(bytes);
  int *a_ref = (int *)alloc_aligned(bytes);
  int *temp = (int *)alloc_aligned((size_t)thread_count * 2 *
                                   (size_t)(n / thread_count) * sizeof(int));

  if (a_orig == NULL || a_work == NULL || a_ref == NULL || temp == NULL) {
    printf("Error: memory allocation failed\n");
    return 1;
  }

  srand(42);
  for (long i = 0; i < n; i++) {
    a_orig[i] = rand() % 100000;
  }

  // Reference result produced by the standard library.
  memcpy(a_ref, a_orig, bytes);
  qsort(a_ref, (size_t)n, sizeof(int), compare_ints);

  double start = 0.0;
  double t[4] = {0.0};
  int ok[4] = {0};

  for (int r = 0; r < NTIMES; r++) {
    memcpy(a_work, a_orig, bytes);
    start = get_time_ms();
    oddeven_serial(a_work, n);
    t[0] += get_time_ms() - start;
  }
  ok[0] = check_equal_int(a_ref, a_work, n);

  for (int r = 0; r < NTIMES; r++) {
    memcpy(a_work, a_orig, bytes);
    start = get_time_ms();
    oddeven_multi_region(a_work, n, thread_count);
    t[1] += get_time_ms() - start;
  }
  ok[1] = check_equal_int(a_ref, a_work, n);

  for (int r = 0; r < NTIMES; r++) {
    memcpy(a_work, a_orig, bytes);
    start = get_time_ms();
    oddeven_single_region(a_work, n, thread_count);
    t[2] += get_time_ms() - start;
  }
  ok[2] = check_equal_int(a_ref, a_work, n);

  for (int r = 0; r < NTIMES; r++) {
    memcpy(a_work, a_orig, bytes);
    start = get_time_ms();
    oddeven_blocked(a_work, n, thread_count, temp);
    t[3] += get_time_ms() - start;
  }
  ok[3] = check_equal_int(a_ref, a_work, n);

  for (int i = 0; i < 4; i++) {
    t[i] /= NTIMES;
  }

  print_table_header();
  print_row("V0: Serial Baseline", t[0], t[0], ok[0]);
  print_row("V1: Multi Parallel Regions", t[1], t[0], ok[1]);
  print_row("V2: Single Parallel Region", t[2], t[0], ok[2]);
  print_row("V3: Blocked Merge-Split", t[3], t[0], ok[3]);
  printf("%s\n", LINE);

  free(a_orig);
  free(a_work);
  free(a_ref);
  free(temp);
  return 0;
}

## 9. 编译与运行

### 9.1 编译

In [ ]:
bin_sort = compile_c('omp_oddeven_sort.c', extra=())

### 9.2 运行

取 $n = 10080$、4 线程。10080 可被 1 至 8 整除，便于后续做线程数扩展性实验而无需更换问题规模。

四个版本的校验均以 `qsort` 的结果为参照，逐元素比对，因而不仅能发现顺序错误，也能发现元素丢失或重复。

In [ ]:
out_sort = run_c(bin_sort, 10080, 4)
rows_sort = parse_table(out_sort)
print()
for name, ms, sp, chk in rows_sort:
    print('  %-28s %9.3f ms  %6.2fx  %s' % (name, ms, sp, chk))

## 10. 结果可视化

In [ ]:
plot_speedup(rows_sort,
             'Lab 5: Odd-Even Transposition Sort (n = 10080, 4 threads)')

### 10.1 Fork-Join 开销的定量估计

V1 与 V2 执行完全相同的算法、完全相同的比较次数、完全相同的栅栏次数，主要差别是线程组的创建次数（$n$ 次对 1 次）。因此两者的耗时之差可用于估计单次 Fork-Join 的代价：

$$t_{\text{fork-join}} \approx \frac{T_{V1} - T_{V2}}{n}$$

严格地说，该差值中还包含线程反复销毁重建所带来的缓存状态与线程亲和性变化。但相对于 Fork-Join 本身，这些因素可视为次要，因此上式在量级上是可靠的。

In [ ]:
byname = {r[0]: r for r in rows_sort}
t_v1 = byname.get('V1: Multi Parallel Regions', (None, float('nan')))[1]
t_v2 = byname.get('V2: Single Parallel Region', (None, float('nan')))[1]
N = 10080

print('V1 (每相位一个并行区域): %9.3f ms' % t_v1)
print('V2 (全程一个并行区域)  : %9.3f ms' % t_v2)
print('差值                   : %9.3f ms' % (t_v1 - t_v2))
print()
print('相位数 n = %d' % N)
print('单次 Fork-Join 代价约为 %.3f 微秒' % ((t_v1 - t_v2) * 1000.0 / N))
print()
print('说明：该估计包含线程组创建、任务分配与汇合的全部代价。')
print('      libgomp 会复用空闲工作线程，因此实测值通常低于')
print('      一次完整的 pthread_create 加 pthread_join。')

## 11. 结果分析

> 以下结论针对**趋势规律**。具体数值随平台、核心数与问题规模而变化。

**① V1 往往慢于串行基准**

这是本实验中最值得重视的现象。V1 的并行化在语法上正确、校验也通过，但其性能仍低于串行基准。

原因在于粒度失配：每个相位的有效计算只有约 $n/2$ 次比较（在 $n=10080$ 时约五千次，耗时数微秒），而为此付出的 Fork-Join 开销也在微秒量级。**当开销与收益处于同一量级时，并行化难以带来实际收益。**

**② V1 → V2 的改进只需移动一行制导语句**

把 `parallel` 从相位循环内部提到外部，线程组创建次数由 $n$ 降为 1，而算法、同步次数、访存模式均未改变。这是一种仅调整结构、不改变算法逻辑的优化，额外的实现成本极低。

**③ V2 仍可能慢于串行**

即便消除了 Fork-Join 开销，V2 仍需 $n$ 次栅栏同步。栅栏的代价虽低于 Fork-Join，但累计 $n$ 次之后依然可观，而每次栅栏之间的有效工作只有约 $n/2$ 次比较。

**这表明：同步开销问题难以仅通过优化同步机制本身解决，而需从算法层面增大同步间隔内的有效工作量。**

**④ V3 的加速比包含算法收益，不能与 V1、V2 直接对比**

这一点必须在教学中明确指出。V3 的加速来自三个方面：

| 来源 | 贡献 |
|---|---|
| 块内算法由 $O(n^2)$ 改为 $O(n\log n)$ | 主要来源 |
| 栅栏次数由 $n$ 降为 $p$ | 次要来源 |
| 多线程并行执行 | 次要来源 |

若要单独衡量 V3 的**并行收益**，应当以「单线程运行 V3」为基准，而非以 V0 为基准。第 12 节的练习 3 即为此设计。

> 在性能报告中应当区分算法改进与并行加速这两类收益。若将算法改进计入并行加速比，会高估并行化本身的贡献，并掩盖真实的并行效率问题。

**⑤ 粒度是并行程序设计的首要考量**

本实验的四个版本给出了一条清晰的递进：

```text
  V1：细粒度并行 + 高频 Fork-Join      → 慢于串行
  V2：细粒度并行 + 一次 Fork-Join      → 接近串行
  V3：粗粒度并行 + 少量同步            → 大幅超越
```

**结论**：面对性能不佳的并行程序，应优先检查并行粒度，其次检查同步方式，最后再考虑微观优化。

## 12. 🔧 动手练习

**练习 1**　把 $n$ 依次取 1008、5040、10080、20160，观察 V1 与 V2 的差值如何随 $n$ 变化。该差值是否与 $n$ 成正比？请解释。

**练习 2**　把线程数依次取 1、2、4、8（$n$ 固定为 10080），观察三个并行版本的扩展性。V3 的加速比在 1 线程时是多少？这个数值说明了什么？

**练习 3**　以「V3 单线程」为基准重新计算 V3 的加速比，把算法收益与并行收益分离开来。

**练习 4**　尝试在 V2 的 `#pragma omp for` 上加 `nowait`，重新编译运行，记录校验结论。请解释为何结果会出错。

**练习 5**（进阶）　在 V2 的相位循环内加入一个 `#pragma omp single` 块，打印当前相位号。观察输出行数是否等于 $n$，并解释 `single` 为何不会导致重复打印。再把 `single` 换成 `master`，比较两者的差异。

**练习 6**（进阶）　V3 的第一步用 `#pragma omp parallel for` 调用 `qsort`。若把它改为 `#pragma omp for schedule(dynamic)`，在块长相等的前提下性能会有变化吗？为什么？

### 12.1 练习 1 的参考实现：Fork-Join 开销随规模的变化

In [ ]:
import matplotlib.pyplot as plt

sizes = [1008, 5040, 10080, 20160]
gap_ms, unit_us = [], []
for n in sizes:
    out = subprocess.run([bin_sort, str(n), '4'],
                         capture_output=True, text=True).stdout
    d = {r[0]: r[1] for r in parse_table(out)}
    gap = d['V1: Multi Parallel Regions'] - d['V2: Single Parallel Region']
    gap_ms.append(gap)
    unit_us.append(gap * 1000.0 / n)
    print('n = %-6d  V1-V2 = %8.3f ms   单次 Fork-Join ≈ %6.3f us'
          % (n, gap, unit_us[-1]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(sizes, gap_ms, marker='o', color='#1f77b4')
ax1.set_xlabel('n (number of phases)')
ax1.set_ylabel('T(V1) - T(V2)  [ms]')
ax1.set_title('Total Fork-Join Overhead')
ax1.grid(linestyle=':', alpha=0.5)
ax2.plot(sizes, unit_us, marker='s', color='#d62728')
ax2.set_xlabel('n (number of phases)')
ax2.set_ylabel('Per-region cost  [us]')
ax2.set_title('Cost of a Single Fork-Join')
ax2.grid(linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()
print()
print('左图应近似为过原点的直线（总开销正比于 n）；')
print('右图应近似水平（单次代价与 n 无关）。')

### 12.2 练习 3 的参考实现：分离算法收益与并行收益

In [ ]:
out1 = subprocess.run([bin_sort, '10080', '1'],
                      capture_output=True, text=True).stdout
out4 = subprocess.run([bin_sort, '10080', '4'],
                      capture_output=True, text=True).stdout
d1 = {r[0]: r[1] for r in parse_table(out1)}
d4 = {r[0]: r[1] for r in parse_table(out4)}

t_serial = d1['V0: Serial Baseline']
t_v3_1 = d1['V3: Blocked Merge-Split']
t_v3_4 = d4['V3: Blocked Merge-Split']

print('V0 串行冒泡        (1 线程): %9.3f ms' % t_serial)
print('V3 块分解          (1 线程): %9.3f ms' % t_v3_1)
print('V3 块分解          (4 线程): %9.3f ms' % t_v3_4)
print()
print('算法收益 = V0(1线程) / V3(1线程) = %6.2fx' % (t_serial / t_v3_1))
print('并行收益 = V3(1线程) / V3(4线程) = %6.2fx' % (t_v3_1 / t_v3_4))
print('合计     = V0(1线程) / V3(4线程) = %6.2fx' % (t_serial / t_v3_4))
print()
print('表格中 V3 一行显示的加速比是「合计」，其中绝大部分来自算法改进。')

## 13. 🤔 思考题

**思考题 1**　V2 中的 `phase` 变量被声明为 `private`。若误写为 `shared`，程序会出现什么现象？除结果错误之外，是否还可能出现其他形式的失效？请从 OpenMP 对工作分担构造的规定出发作答。

**思考题 2**　V2 的相位循环由每个线程完整执行一遍，这是否意味着比较交换的工作也被重复执行了 $p$ 遍？请指出是哪一处结构避免了重复。

**思考题 3**　若把 V2 中的两个 `#pragma omp for` 都加上 `nowait`，同时在相位循环末尾手工加一个 `#pragma omp barrier`，程序是否正确？性能与原版相比会如何？

**思考题 4**　V3 的第二步使用了 `thread_count` 个相位。请证明：对 $p$ 个已各自有序的块，$p$ 个奇偶相位足以使整体有序。

**思考题 5**　V3 为每个线程分配了 $2 \times \text{local\_n}$ 个整数的缓冲区。若改为所有线程共用一个缓冲区并以 `critical` 保护，正确性能否保证？性能会如何变化？

**思考题 6**（综合）　本实验表明「粗粒度并行优于细粒度并行」。但粒度也不是越粗越好——请举出一个粒度过粗反而不利的场景，并说明其代价来自何处。（提示：结合实验四的负载均衡。）

## 14. 📌 本实验小结

| 概念 | 要点 |
|---|---|
| `parallel` 与 `for` 分离 | 线程组创建一次，可被多次使用 |
| Fork-Join 开销 | 微秒量级，乘以高频次数后可超过有效计算 |
| 隐式栅栏 | `for` / `sections` / `single` 末尾有，`parallel` 末尾必有 |
| `nowait` | 仅在后续代码不读取本循环写入的数据时可用 |
| `single` | 任一线程执行，**有**隐式栅栏 |
| `master` | 必为 0 号线程执行，**无**隐式栅栏；OpenMP 5.1 起弃用，由 `masked` 取代 |
| 并行粒度 | 同步之间的工作量必须显著大于同步开销 |
| 收益分离 | 算法收益与并行收益应分别报告，不可混淆 |

### 一条可直接使用的判断

> 若某个并行版本的性能低于串行版本，应首先估算：**每次同步之间的有效计算量是多少微秒？**若该量与同步开销处于同一量级，则问题主要出在并行粒度上，此时微观优化往往收效有限。

### 与后续实验的衔接

本实验讨论的是**结构性**开销——线程组创建与栅栏。下一个实验转向**数据访问**层面的开销：当多个线程需要更新同一批数据时，互斥的代价有多大，以及一种更隐蔽的代价——伪共享。